# 01 数据读取与质量核验

目标：读取三份样本，核对数据结构、质量标记及站点匹配。
此阶段不填补 AQHI、不训练模型，也不把历史快照当作实时数据。
每个代码单元先阅读、再运行，并检查输出。


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print  # Allows verification outside a Jupyter kernel.

# Supports launching Jupyter from either the project root or notebooks directory.
ROOT = Path.cwd()
if not (ROOT / 'src' / 'aqhi_data.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'aqhi_data.py').exists(), '请在项目目录启动 Jupyter'
sys.path.insert(0, str(ROOT / 'src'))
from aqhi_data import read_history, read_current, read_network, audit, parse_aqhi
RAW = ROOT / 'data' / 'raw'
display(pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size} for p in RAW.iterdir() if p.is_file()]))


,file,bytes
0,202606_ChS.csv,32553
1,Air_Quality_Monitoring_Network_of_Hong_Kong_GE...,3141
2,aqhi_ind_rss_ChS.xml,7127


## 1. 读取历史数据

自动定位“日期,时间”表头。日期空白仅代表沿用上一行日期，因此只前向填充日期，
不填充 AQHI。宽表转换成长表后，每行表示一个站点在一个小时的记录。
原 CSV 每天另含“每日最高值”汇总行，排除这些行并记录数量，避免把它们当成小时观测。
保留原日期和小时；24:00 暂按下一天 00:00 解释，建模前需核对官方字典。


In [2]:
history = read_history(RAW / '202606_ChS.csv')
display(history.head())
display(history[['source_date', 'source_hour', 'timestamp']].drop_duplicates().head(25))
print('行数与列数:', history.shape)


,source_date,source_hour,timestamp,station_source,station_id,aqhi_raw,aqhi_value,is_substituted,is_10_plus,is_missing,is_unavailable,is_invalid,risk_category,source_file
0,2026-06-01,01,2026-06-01 01:00:00+08:00,中西区,Central/Western,5,5.0,False,False,False,False,False,Moderate,202606_ChS.csv
1,2026-06-01,02,2026-06-01 02:00:00+08:00,中西区,Central/Western,5,5.0,False,False,False,False,False,Moderate,202606_ChS.csv
2,2026-06-01,03,2026-06-01 03:00:00+08:00,中西区,Central/Western,5,5.0,False,False,False,False,False,Moderate,202606_ChS.csv
3,2026-06-01,04,2026-06-01 04:00:00+08:00,中西区,Central/Western,5,5.0,False,False,False,False,False,Moderate,202606_ChS.csv
4,2026-06-01,05,2026-06-01 05:00:00+08:00,中西区,Central/Western,5,5.0,False,False,False,False,False,Moderate,202606_ChS.csv


,source_date,source_hour,timestamp
0,2026-06-01,01,2026-06-01 01:00:00+08:00
1,2026-06-01,02,2026-06-01 02:00:00+08:00
2,2026-06-01,03,2026-06-01 03:00:00+08:00
3,2026-06-01,04,2026-06-01 04:00:00+08:00
4,2026-06-01,05,2026-06-01 05:00:00+08:00
5,2026-06-01,06,2026-06-01 06:00:00+08:00
6,2026-06-01,07,2026-06-01 07:00:00+08:00
7,2026-06-01,08,2026-06-01 08:00:00+08:00
8,2026-06-01,09,2026-06-01 09:00:00+08:00
9,2026-06-01,10,2026-06-01 10:00:00+08:00


行数与列数: (12960, 14)


## 2. 检查特殊值

`*` 表示官方使用邻近站数据替代计算，并非我们填补的值。
`10+` 保留为截顶类别，其 aqhi_value 为空；通过 is_10_plus 区别于真正缺失。
未知格式标记为 is_invalid，必须人工检查，不静默转换。
样本中的 `-` / `-*` 单独记录为 is_unavailable，不当作 0 或有效观测；具体来源含义待字典核对。


In [3]:
display(pd.DataFrame([parse_aqhi(x) for x in ['', '4*', '10+', '7', 'bad']]))
display(history.groupby('station_id')[['is_missing', 'is_unavailable', 'is_substituted', 'is_10_plus', 'is_invalid']].sum())
display(history[history.is_invalid].head(10))


,aqhi_raw,aqhi_value,is_substituted,is_10_plus,is_missing,is_unavailable,is_invalid,risk_category
0,,NaN,False,False,True,False,False,NaN
1,4*,4.0,True,False,False,False,False,Moderate
2,10+,NaN,False,True,False,False,False,Serious
3,7,7.0,False,False,False,False,False,High
4,bad,NaN,False,False,False,False,True,NaN


,is_missing,is_unavailable,is_substituted,is_10_plus,is_invalid
station_id,,,,,
Causeway Bay,3,5,12,0,0
Central,3,23,195,0,0
Central/Western,3,5,4,0,0
Eastern,3,5,21,0,0
Kwai Chung,3,5,11,0,0
Kwun Tong,3,5,15,0,0
Mong Kok,3,5,7,0,0
North,3,5,59,0,0
Sha Tin,3,5,9,0,0


,source_date,source_hour,timestamp,station_source,station_id,aqhi_raw,aqhi_value,is_substituted,is_10_plus,is_missing,is_unavailable,is_invalid,risk_category,source_file


## 3. 读取监测站网络

直接读取 ZIP 内的 GeoJSON，无需改动原压缩包。核对几何坐标与属性经纬度一致。
使用明确的简繁站名映射；无法识别的站名会报错，而不会模糊匹配。


In [4]:
network = read_network(RAW / 'Air_Quality_Monitoring_Network_of_Hong_Kong_GEOJSON.zip')
display(network)
display(network.station_type.value_counts())


,station_id,name_en,name_tc,station_type,longitude,latitude,source_last_update
0,Tsuen Wan,Tsuen Wan Air Quality Monitoring Station,荃灣空氣質素監測站,GENERAL MONITORING STATION,114.114535,22.371742,20260204103016
1,Causeway Bay,Causeway Bay Air Quality Monitoring Station,銅鑼灣空氣質素監測站,ROADSIDE MONITORING STATION,114.185090,22.280133,20260204102933
2,Central,Central Air Quality Monitoring Station,中環空氣質素監測站,ROADSIDE MONITORING STATION,114.158127,22.281815,20260204102851
3,Central/Western,Central/Western Air Quality Monitoring Station,中西區空氣質素監測站,GENERAL MONITORING STATION,114.144421,22.284891,20260204102803
4,Mong Kok,Mong Kok Air Quality Monitoring Station,旺角空氣質素監測站,ROADSIDE MONITORING STATION,114.168272,22.322611,20260204102724
5,Tai Po,Tai Po Air Quality Monitoring Station,大埔空氣質素監測站,GENERAL MONITORING STATION,114.164570,22.450960,20260204102647
6,Sham Shui Po,Sham Shui Po Air Quality Monitoring Station,深水埗空氣質素監測站,GENERAL MONITORING STATION,114.159109,22.330226,20260204102535
7,Sha Tin,Sha Tin Air Quality Monitoring Station,沙田空氣質素監測站,GENERAL MONITORING STATION,114.184532,22.376281,20260204102448
8,Yuen Long,Yuen Long Air Quality Monitoring Station,元朗空氣質素監測站,GENERAL MONITORING STATION,114.022649,22.445155,20260204102409
9,Kwai Chung,Kwai Chung Air Quality Monitoring Station,葵涌空氣質素監測站,GENERAL MONITORING STATION,114.129601,22.357104,20260204102321


station_type
GENERAL MONITORING STATION     15
ROADSIDE MONITORING STATION     3
Name: count, dtype: int64

## 4. 读取保存的 RSS 快照

这个 XML 不会自动更新。published_at 是 RSS 中的 pubDate，先保留其原始语义。
此处不把 RSS 与六月历史记录按时间拼接；只通过站名关联地理信息。


In [5]:
current = read_current(RAW / 'aqhi_ind_rss_ChS.xml')
display(current[['station_id', 'aqhi_raw', 'official_risk_text', 'published_at']])


,station_id,aqhi_raw,official_risk_text,published_at
0,Central/Western,4,中,2026-11-02 12:30:00+08:00
1,Southern,4,中,2026-11-02 12:30:00+08:00
2,Eastern,4,中,2026-11-02 12:30:00+08:00
3,Kwun Tong,4,中,2026-11-02 12:30:00+08:00
4,Sham Shui Po,3,低,2026-11-02 12:30:00+08:00
5,Kwai Chung,4,中,2026-11-02 12:30:00+08:00
6,Tsuen Wan,4,中,2026-11-02 12:30:00+08:00
7,Tseung Kwan O,3,低,2026-11-02 12:30:00+08:00
8,Yuen Long,4,中,2026-11-02 12:30:00+08:00
9,Tuen Mun,4,中,2026-11-02 12:30:00+08:00


## 5. 质量报告与关联

检查重复站点小时、未知数值、时间缺口及无法匹配的站点。
重复记录需要调查原因，不能直接删除。时间缺口仅检查样本已观测范围。
18 个站点匹配成功不等于数据覆盖或预测效果已经得到验证。


In [6]:
summary = audit(history, current, network)
print(json.dumps(summary, ensure_ascii=False, indent=2))
history_with_stations = history.merge(network, on='station_id', how='left', validate='many_to_one')
current_with_stations = current.merge(network, on='station_id', how='left', validate='many_to_one')
assert summary['invalid_values'] == 0, '先调查未知 AQHI 格式'
assert summary['duplicate_station_hours'] == 0, '先调查重复记录'
assert summary['history_unmatched'] == summary['current_unmatched'] == 0, '先处理站点匹配问题'
display(current_with_stations[['station_id', 'aqhi_raw', 'station_type', 'longitude', 'latitude']])


{
  "history_rows": 12960,
  "history_stations": 18,
  "excluded_daily_summary_rows": 30,
  "history_start": "2026-06-01 01:00:00+08:00",
  "history_end": "2026-07-01 00:00:00+08:00",
  "missing_values": 54,
  "unavailable_dash_values": 138,
  "substituted_values": 1108,
  "capped_values": 0,
  "invalid_values": 0,
  "duplicate_station_hours": 0,
  "missing_hours_within_observed_range": {
    "Causeway Bay": 0,
    "Central": 0,
    "Central/Western": 0,
    "Eastern": 0,
    "Kwai Chung": 0,
    "Kwun Tong": 0,
    "Mong Kok": 0,
    "North": 0,
    "Sha Tin": 0,
    "Sham Shui Po": 0,
    "Southern": 0,
    "Tai Po": 0,
    "Tap Mun": 0,
    "Tseung Kwan O": 0,
    "Tsuen Wan": 0,
    "Tuen Mun": 0,
    "Tung Chung": 0,
    "Yuen Long": 0
  },
  "network_stations": 18,
  "current_stations": 18,
  "current_published_at": [
    "2026-11-02 12:30:00+08:00"
  ],
  "history_unmatched": 0,
  "current_unmatched": 0
}


,station_id,aqhi_raw,station_type,longitude,latitude
0,Central/Western,4,GENERAL MONITORING STATION,114.144421,22.284891
1,Southern,4,GENERAL MONITORING STATION,114.160140,22.247461
2,Eastern,4,GENERAL MONITORING STATION,114.219372,22.282886
3,Kwun Tong,4,GENERAL MONITORING STATION,114.231174,22.309625
4,Sham Shui Po,3,GENERAL MONITORING STATION,114.159109,22.330226
5,Kwai Chung,4,GENERAL MONITORING STATION,114.129601,22.357104
6,Tsuen Wan,4,GENERAL MONITORING STATION,114.114535,22.371742
7,Tseung Kwan O,3,GENERAL MONITORING STATION,114.259561,22.317642
8,Yuen Long,4,GENERAL MONITORING STATION,114.022649,22.445155
9,Tuen Mun,4,GENERAL MONITORING STATION,113.976728,22.391143


## 6. 导出可追溯的派生数据

本单元会创建/更新 processed 和 reports 中的结果，绝不修改 raw。
这些是保留质量标记的样本整理结果，还不是最终训练数据。
下一步：补齐历史月份，核验时间字典和站点沿革，再设计训练/验证/测试划分。


In [7]:
OUT = ROOT / 'data' / 'processed'
REPORT = ROOT / 'reports'
OUT.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)
history_with_stations.to_csv(OUT / 'history_202606_audited.csv', index=False, encoding='utf-8-sig')
current_with_stations.to_csv(OUT / 'current_snapshot_audited.csv', index=False, encoding='utf-8-sig')
network.to_csv(OUT / 'stations.csv', index=False, encoding='utf-8-sig')
(REPORT / 'sample_audit.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('已导出样本数据与质量报告：', OUT, REPORT)


已导出样本数据与质量报告： C:\Users\hxq08259\Desktop\sem1\Math7005\data\processed C:\Users\hxq08259\Desktop\sem1\Math7005\reports
